# 12) Presentation Summary (Slide-ready)

このノートブックは、本研究（リスク開示の新規性 → 市場反応）の **発表用まとめ**です。

## 目的
- リスク開示テキストの **新規性（chunk-based novelty）** を作り、イベントスタディで市場反応を検証する。

## このノートブックが作る“スライド素材”
- **図（PNG）**
  - `slides/figures/event_time_abvol.png`
  - `slides/figures/event_time_abs_ar.png`
- **表（CSV）**
  - `slides/tables/regression_summary.csv`

## 実行方法
- 上から順に **Run All**。
- 既に `risk_change_scores` がDBに存在する前提。

> 注：実行時間を短くするため、既定ではサンプル（例：最大 8,000 件）で回します。発表前に全件で回したい場合は `SAMPLE_N` を増やすか、`SAMPLE_MODE=False` にしてください。



In [ ]:
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

import matplotlib.pyplot as plt

# optional (OLS)
try:
    import statsmodels.api as sm
except Exception:
    sm = None

plt.rcParams["figure.figsize"] = (10, 4)



In [ ]:
def find_project_root(start_dir: str) -> str:
    d = os.path.abspath(start_dir)
    while True:
        if os.path.exists(os.path.join(d, "docker-compose.yml")) and os.path.exists(os.path.join(d, "requirements.txt")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            return os.path.abspath(start_dir)
        d = parent


PROJECT_ROOT = find_project_root(os.getcwd())
os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

load_dotenv(dotenv_path=os.path.join(PROJECT_ROOT, ".env"))
DB_USER = os.getenv("POSTGRES_USER")
DB_PASS = os.getenv("POSTGRES_PASSWORD")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")
DB_NAME = os.getenv("POSTGRES_DB")

engine = create_engine(f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print("DB connected")

DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
SLIDES_DIR = os.path.join(PROJECT_ROOT, "slides")
FIG_DIR = os.path.join(SLIDES_DIR, "figures")
TABLE_DIR = os.path.join(SLIDES_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

TOPIX500_CATEGORIES = {"TOPIX Core30", "TOPIX Large70", "TOPIX Mid400"}

# --- Config (presentation default) ---
SAMPLE_MODE = True
SAMPLE_N = 8000
SAMPLE_SEED = 42

EVENT_WINDOWS = {
    "m1_p1": (-1, 1),
    "0_p1": (0, 1),
    "m3_p3": (-3, 3),
}
EST_WINDOW = (-120, -21)
CUTOFF_HOUR = 15
MIN_EST_OBS = 60

NOVELTY_COL = os.getenv("EVENT_NOVELTY_COL", "change_topk")
TOPIX500_ONLY = False

print("SAMPLE_MODE=", SAMPLE_MODE, "SAMPLE_N=", SAMPLE_N, "NOVELTY_COL=", NOVELTY_COL)



PROJECT_ROOT: /home/junxzi/dev/risk-novelty-stock-jp
DB connected
SAMPLE_MODE= True SAMPLE_N= 8000 NOVELTY_COL= change_topk


In [ ]:
def load_topix_membership(years: list[int]) -> dict[int, list[str]]:
    out: dict[int, list[str]] = {}
    for y in years:
        p = os.path.join(DATA_DIR, f"topix_companies_{y}.csv")
        m = pd.read_csv(p)
        m["security_code"] = m["security_code"].astype(str).str.zfill(5)
        if "scale_category" in m.columns:
            m = m[m["scale_category"].isin(TOPIX500_CATEGORIES)]
        out[y] = sorted(m["security_code"].dropna().unique().tolist())
    return out


def load_company_snapshot(years: list[int]) -> pd.DataFrame:
    frames = []
    for y in years:
        p = os.path.join(DATA_DIR, f"topix_companies_{y}.csv")
        t = pd.read_csv(p)
        t["security_code"] = t["security_code"].astype(str).str.zfill(5)
        t["year"] = int(y)
        frames.append(t)
    snap = pd.concat(frames, ignore_index=True)
    keep = ["year", "security_code"]
    for c in ["sector_17", "sector_33", "scale_category", "market_code", "name_ja"]:
        if c in snap.columns:
            keep.append(c)
    return snap[keep].drop_duplicates(["year", "security_code"])


def build_market_return(quotes: pd.DataFrame, membership_by_year: dict[int, list[str]]) -> pd.Series:
    rm_frames = []
    for y, codes in membership_by_year.items():
        sub = quotes[quotes["code"].isin(codes)][["date", "ret"]].dropna()
        if sub.empty:
            continue
        rm = sub.groupby("date")["ret"].mean().rename("rm").reset_index()
        rm["year"] = y
        rm_frames.append(rm)
    rm = pd.concat(rm_frames, ignore_index=True).sort_values("date")
    rm = rm.drop_duplicates("date", keep="last")
    return rm.set_index("date")["rm"]


def next_td(g: pd.DataFrame, day: pd.Timestamp) -> pd.Timestamp | None:
    i = g["date"].searchsorted(day, side="left")
    if i >= len(g):
        return None
    return g.loc[i, "date"]


def shift_td(g: pd.DataFrame, day: pd.Timestamp, k: int) -> pd.Timestamp | None:
    i = g["date"].searchsorted(day, side="left")
    if i >= len(g):
        return None
    j = i + k
    if j < 0 or j >= len(g):
        return None
    return g.loc[j, "date"]


def estimate_alpha_beta(stock_ret: pd.DataFrame, rm: pd.Series, event_day: pd.Timestamp, est_window: tuple[int, int], min_obs: int):
    s = shift_td(stock_ret, event_day, est_window[0])
    e = shift_td(stock_ret, event_day, est_window[1])
    if s is None or e is None:
        return None
    est = stock_ret[(stock_ret["date"] >= s) & (stock_ret["date"] <= e)].copy()
    est = est.join(rm, on="date", how="inner").dropna(subset=["ret", "rm"])
    if len(est) < min_obs:
        return None
    x = est["rm"].to_numpy()
    y = est["ret"].to_numpy()
    X = np.column_stack([np.ones_like(x), x])
    a, b = np.linalg.lstsq(X, y, rcond=None)[0]
    return float(a), float(b)


def event_window_panel(
    stock_df: pd.DataFrame,
    rm: pd.Series,
    submit_ts: pd.Timestamp,
    window: tuple[int, int],
    cutoff_hour: int,
    est_window: tuple[int, int],
    min_est_obs: int,
):
    """Build event-window panel with columns: date, ret, rm, ar (+ volume fields if available)."""
    base = pd.Timestamp(submit_ts.date())
    ev = next_td(stock_df, base)
    if ev is None:
        return None
    if submit_ts.hour >= cutoff_hour:
        ev = shift_td(stock_df, ev, 1)
        if ev is None:
            return None

    ab = estimate_alpha_beta(stock_df[["date", "ret"]].dropna().reset_index(drop=True), rm, ev, est_window, min_est_obs)
    if ab is None:
        return None
    a, b = ab

    # stock-specific expected log-volume estimated in the estimation window
    s_est = shift_td(stock_df, ev, est_window[0])
    e_est = shift_td(stock_df, ev, est_window[1])
    mu_log_vol = None
    if s_est is not None and e_est is not None and "log_vol" in stock_df.columns:
        est_vol = stock_df[(stock_df["date"] >= s_est) & (stock_df["date"] <= e_est)]["log_vol"].dropna()
        if len(est_vol) >= min_est_obs:
            mu_log_vol = float(est_vol.mean())

    s = shift_td(stock_df, ev, window[0])
    e = shift_td(stock_df, ev, window[1])
    if s is None or e is None:
        return None

    cols = ["date", "ret"]
    for c in ["adjusted_volume", "log_vol", "avol"]:
        if c in stock_df.columns:
            cols.append(c)

    evdf = stock_df[(stock_df["date"] >= s) & (stock_df["date"] <= e)][cols].copy()
    evdf = evdf.join(rm, on="date", how="inner").dropna(subset=["ret", "rm"])
    if evdf.empty:
        return None

    evdf["ar"] = evdf["ret"] - (a + b * evdf["rm"])

    # abnormal log-volume vs stock baseline
    if mu_log_vol is not None and "log_vol" in evdf.columns:
        evdf["ablogvol_stock"] = evdf["log_vol"] - mu_log_vol

    return evdf


def summarize_uncertainty(evdf: pd.DataFrame):
    ar = evdf["ar"].astype(float)
    out = {
        "car_mm": float(ar.sum()),
        "vol_abs": float(np.abs(ar).sum()),
        "vol_sq": float((ar**2).sum()),
        "downside": float(ar.min()),
        "abvol_mkt": float(evdf["avol"].dropna().sum()) if "avol" in evdf.columns else np.nan,
        "abvol_stock": float(evdf["ablogvol_stock"].dropna().sum()) if "ablogvol_stock" in evdf.columns else np.nan,
        "n_days": int(len(evdf)),
    }
    return out


def safe_log(x):
    x = pd.to_numeric(x, errors="coerce")
    return np.log(x.replace(0, np.nan))


def run_ols(base: pd.DataFrame, y_col: str, label: str):
    if sm is None:
        raise RuntimeError("statsmodels is not available")

    sub = base.dropna(subset=[y_col, "novelty", "log_assets", "log_revenue", "roa", "lev"]).copy()
    if len(sub) < 300:
        print("[ols]", label, "skip n=", len(sub))
        return None

    y = pd.to_numeric(sub[y_col], errors="coerce").astype(float)
    X = sub[["novelty", "log_assets", "log_revenue", "roa", "lev"]].apply(pd.to_numeric, errors="coerce")
    X = pd.concat([X, pd.get_dummies(sub["fiscal_year"].astype(int), prefix="fy", drop_first=True, dtype=float)], axis=1)
    X = pd.concat([X, pd.get_dummies(sub["sector_17"].fillna("unknown").astype(str), prefix="sec", drop_first=True, dtype=float)], axis=1)
    X = sm.add_constant(X, has_constant="add")

    reg = pd.concat([y.rename("y"), X], axis=1).dropna()
    y2 = reg["y"]
    X2 = reg.drop(columns=["y"])

    res = sm.OLS(y2, X2).fit(cov_type="HC1")
    b = res.params.get("novelty", np.nan)
    t = res.tvalues.get("novelty", np.nan)
    p = res.pvalues.get("novelty", np.nan)
    print(f"[ols] {label} n={len(reg)} beta={b:.4g} t={t:.3g} p={p:.3g} r2={res.rsquared:.3g}")
    return {
        "label": label,
        "y": y_col,
        "n": int(len(reg)),
        "beta_novelty": float(b) if b == b else np.nan,
        "t_novelty": float(t) if t == t else np.nan,
        "p_novelty": float(p) if p == p else np.nan,
        "r2": float(res.rsquared),
    }



In [ ]:
# --- 1) Load risk scores + meta (submit_ts, security_code) ---

# Select novelty column dynamically (supports category-specific novelty columns)
base_cols = [
    "company_id::uuid as company_id",
    "fiscal_year",
    "doc_id_curr",
    "change_topk",
    "change_mean",
    "change_ratio",
    "max_sim_p10",
    "max_sim_median",
]
if NOVELTY_COL not in {"change_topk", "change_mean", "change_ratio"}:
    base_cols.append(NOVELTY_COL)

scores_sql = "select " + ", ".join(base_cols) + " from risk_change_scores"
scores = pd.read_sql(scores_sql, engine)
print("risk_change_scores:", len(scores))

if SAMPLE_MODE:
    scores = scores.sample(n=min(SAMPLE_N, len(scores)), random_state=SAMPLE_SEED).copy()
    print("sampled:", len(scores))

meta = pd.read_sql(
    text(
        """
        select d.doc_id, d.submit_date, c.security_code
        from edinet_documents d
        join companies c on c.company_id = d.company_id
        where d.doc_id = any(:doc_ids)
        """
    ),
    engine,
    params={"doc_ids": scores["doc_id_curr"].tolist()},
)
meta["submit_ts"] = pd.to_datetime(meta["submit_date"])
meta["security_code"] = meta["security_code"].astype(str).str.zfill(5)

base = scores.merge(meta[["doc_id", "submit_ts", "security_code"]], left_on="doc_id_curr", right_on="doc_id", how="left")
base = base.dropna(subset=["submit_ts", "security_code"]).copy()
base["year"] = base["submit_ts"].dt.year

# --- 2) Controls ---
controls = pd.read_sql(
    """
    select company_id::uuid as company_id, fiscal_year,
           total_assets, total_liabilities, net_assets, revenue, operating_income, net_income
    from edinet_controls
    """,
    engine,
)
base = base.merge(controls, on=["company_id", "fiscal_year"], how="left")

# --- 3) Snapshot meta (sector, scale) ---
needed_years = sorted(base["year"].dropna().astype(int).unique().tolist())
print("needed_years:", needed_years)

membership_by_year = load_topix_membership(needed_years)
snap = load_company_snapshot(needed_years)
base = base.merge(snap, on=["year", "security_code"], how="left")

if TOPIX500_ONLY:
    base = base[base["scale_category"].isin(TOPIX500_CATEGORIES)].copy()
    print("[filter] TOPIX500 only -> docs:", len(base))

# unify novelty field
if NOVELTY_COL not in base.columns:
    raise ValueError(f"NOVELTY_COL={NOVELTY_COL} not found in columns")
base["novelty"] = pd.to_numeric(base[NOVELTY_COL], errors="coerce")

print("base docs:", len(base), "unique doc_id:", base["doc_id_curr"].nunique())
base[["novelty", "fiscal_year", "year"]].describe()



risk_change_scores: 7336
sampled: 7336
needed_years: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
base docs: 7336 unique doc_id: 7336


,novelty,fiscal_year,year
count,7.336000e+03,7336.000000,7336.000000
mean,1.863543e-01,2021.731189,2021.731189
std,1.327330e-01,2.243947,2.243947
min,-1.192093e-07,2018.000000,2018.000000
25%,5.742448e-02,2020.000000,2020.000000
50%,1.951169e-01,2022.000000,2022.000000
75%,2.666256e-01,2024.000000,2024.000000
max,6.719558e-01,2025.000000,2025.000000


In [ ]:
# --- 4) Load daily quotes (for stock + market) and build rm, avol ---

min_day = base["submit_ts"].dt.floor("D").min() - pd.Timedelta(days=450)
max_day = base["submit_ts"].dt.floor("D").max() + pd.Timedelta(days=30)

stock_codes = sorted(base["security_code"].unique().tolist())
market_codes = sorted(set(sum((membership_by_year[y] for y in needed_years), [])))
all_codes = sorted(set(stock_codes) | set(market_codes))

print("date range:", min_day.date(), "..", max_day.date())
print("stock_codes:", len(stock_codes), "market_codes:", len(market_codes), "all:", len(all_codes))

quotes = pd.read_sql(
    text(
        """
        select code, date, adjusted_close, adjusted_volume
        from daily_quotes
        where code = any(:codes) and date between :d0 and :d1
        order by code, date
        """
    ),
    engine,
    params={"codes": all_codes, "d0": min_day.date(), "d1": max_day.date()},
)
quotes["code"] = quotes["code"].astype(str).str.zfill(5)
quotes["date"] = pd.to_datetime(quotes["date"])
quotes["adjusted_close"] = pd.to_numeric(quotes["adjusted_close"], errors="coerce")
quotes["adjusted_volume"] = pd.to_numeric(quotes["adjusted_volume"], errors="coerce")
quotes = quotes.dropna(subset=["adjusted_close"]).sort_values(["code", "date"]).copy()
quotes["ret"] = quotes.groupby("code")["adjusted_close"].pct_change()
quotes["log_vol"] = np.log(quotes["adjusted_volume"].where(quotes["adjusted_volume"] > 0))

# market mean volume (cross-sectional baseline)
mkt_vol = quotes[quotes["code"].isin(market_codes)][["date", "adjusted_volume"]].dropna()
if not mkt_vol.empty:
    mkt_mean_vol = mkt_vol.groupby("date")["adjusted_volume"].mean().rename("mkt_mean_vol")
else:
    mkt_mean_vol = pd.Series(dtype=float, name="mkt_mean_vol")

# stock panels (ret + volume fields)
stock_by = {}
for c, g in quotes[quotes["code"].isin(stock_codes)].groupby("code"):
    gg = g[["date", "ret", "adjusted_volume", "log_vol"]].copy()
    if not mkt_mean_vol.empty:
        gg = gg.join(mkt_mean_vol, on="date", how="left")
        gg["avol"] = gg["log_vol"] - np.log(gg["mkt_mean_vol"])
    stock_by[c] = gg.dropna(subset=["ret"]).reset_index(drop=True)

rm = build_market_return(quotes.dropna(subset=["ret"]).copy(), membership_by_year)
print("rm dates:", rm.index.min().date(), "..", rm.index.max().date(), "n=", len(rm))
print("stock panels:", len(stock_by))



date range: 2016-11-01 .. 2025-07-30
stock_codes: 1194 market_codes: 583 all: 1195
rm dates: 2016-11-02 .. 2025-07-02 n= 2116
stock panels: 1194


In [ ]:
# --- 5) Build event-study dataset (with caching) ---

CACHE_TAG = f"{NOVELTY_COL}_sample{int(SAMPLE_MODE)}_n{len(base)}_seed{SAMPLE_SEED}_topix{int(TOPIX500_ONLY)}"
CACHE_PATH = os.path.join(DATA_DIR, f"presentation_event_dataset_{CACHE_TAG}.parquet")
print("CACHE_PATH:", CACHE_PATH)

if os.path.exists(CACHE_PATH):
    out = pd.read_parquet(CACHE_PATH)
    print("loaded cached dataset:", len(out))
else:
    rows = []
    t0 = pd.Timestamp.utcnow()
    for i, r in enumerate(base.itertuples(index=False), start=1):
        code = str(r.security_code)
        stock_ret = stock_by.get(code)
        if stock_ret is None or stock_ret.empty:
            continue
        submit_ts = pd.to_datetime(r.submit_ts)

        for wname, win in EVENT_WINDOWS.items():
            evdf = event_window_panel(
                stock_df=stock_ret,
                rm=rm,
                submit_ts=submit_ts,
                window=win,
                cutoff_hour=CUTOFF_HOUR,
                est_window=EST_WINDOW,
                min_est_obs=MIN_EST_OBS,
            )
            if evdf is None or evdf.empty:
                continue
            unc = summarize_uncertainty(evdf)
            rows.append(
                {
                    "company_id": r.company_id,
                    "fiscal_year": int(r.fiscal_year),
                    "doc_id_curr": r.doc_id_curr,
                    "security_code": code,
                    "sector_17": getattr(r, "sector_17", None),
                    "scale_category": getattr(r, "scale_category", None),
                    "submit_ts": submit_ts,
                    "year": int(r.year),
                    "window": wname,
                    "novelty": float(getattr(r, "novelty")),
                    "novelty_col": NOVELTY_COL,
                    # outcomes
                    "car_mm": unc["car_mm"],
                    "vol_abs": unc["vol_abs"],
                    "vol_sq": unc["vol_sq"],
                    "downside": unc["downside"],
                    "abvol_mkt": unc["abvol_mkt"],
                    "abvol_stock": unc["abvol_stock"],
                    "n_days": unc["n_days"],
                    # controls
                    "total_assets": r.total_assets,
                    "total_liabilities": r.total_liabilities,
                    "net_assets": r.net_assets,
                    "revenue": r.revenue,
                    "operating_income": r.operating_income,
                    "net_income": r.net_income,
                }
            )

        if i % 500 == 0:
            dt = (pd.Timestamp.utcnow() - t0).total_seconds() / 60
            print(f"progress {i}/{len(base)} ({i/len(base):.1%}) elapsed={dt:.1f}min rows={len(rows)}")

    out = pd.DataFrame(rows)
    out.to_parquet(CACHE_PATH, index=False)
    print("saved dataset:", len(out), "->", CACHE_PATH)

print("dataset rows:", len(out), "unique docs:", out["doc_id_curr"].nunique())
out.head(3)



CACHE_PATH: /home/junxzi/dev/risk-novelty-stock-jp/data/processed/presentation_event_dataset_change_topk_sample1_n7336_seed42_topix0.parquet
progress 500/7336 (6.8%) elapsed=0.1min rows=1493
progress 1000/7336 (13.6%) elapsed=0.2min rows=2980
progress 1500/7336 (20.4%) elapsed=0.3min rows=4468
progress 2000/7336 (27.3%) elapsed=0.4min rows=5948
progress 2500/7336 (34.1%) elapsed=0.5min rows=7437
progress 3000/7336 (40.9%) elapsed=0.5min rows=8906
progress 3500/7336 (47.7%) elapsed=0.6min rows=10396
progress 4000/7336 (54.5%) elapsed=0.7min rows=11888
progress 4500/7336 (61.3%) elapsed=0.8min rows=13366
progress 5000/7336 (68.2%) elapsed=0.9min rows=14851
progress 5500/7336 (75.0%) elapsed=1.0min rows=16344
progress 6000/7336 (81.8%) elapsed=1.1min rows=17829
progress 6500/7336 (88.6%) elapsed=1.2min rows=19322
progress 7000/7336 (95.4%) elapsed=1.3min rows=20802


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.

In [ ]:
# --- 6) Regression summary (slide table) ---

# controls / transforms
out2 = out.copy()
out2["log_assets"] = safe_log(out2["total_assets"])
out2["log_revenue"] = safe_log(out2["revenue"])
out2["roa"] = pd.to_numeric(out2["net_income"], errors="coerce") / pd.to_numeric(out2["total_assets"], errors="coerce")
out2["lev"] = pd.to_numeric(out2["total_liabilities"], errors="coerce") / pd.to_numeric(out2["total_assets"], errors="coerce")

MAIN_W = "m1_p1" if "m1_p1" in out2["window"].unique().tolist() else sorted(out2["window"].unique().tolist())[0]
sub = out2[out2["window"] == MAIN_W].copy()
print("MAIN_W:", MAIN_W, "rows:", len(sub), "unique docs:", sub["doc_id_curr"].nunique())

rows = []
if sm is None:
    print("statsmodels is not available -> skip OLS summary")
else:
    for y in ["car_mm", "vol_abs", "abvol_mkt", "abvol_stock", "downside"]:
        r = run_ols(sub, y_col=y, label=f"{MAIN_W}|{y}")
        if r is not None:
            rows.append(r)

regsum = pd.DataFrame(rows)
regsum["novelty_col"] = NOVELTY_COL
regsum["window"] = MAIN_W

OUT_CSV = os.path.join(TABLE_DIR, "regression_summary.csv")
regsum.to_csv(OUT_CSV, index=False)
print("wrote:", OUT_CSV)

regsum



In [ ]:
# --- 7) Slide figures (event-time pattern) ---

PLOT_SAMPLE_N = 500
PLOT_SEED = 42
WIN = (-3, 3)

plot_base = base.dropna(subset=["novelty", "submit_ts", "security_code"]).copy()
plot_base = plot_base.sample(n=min(PLOT_SAMPLE_N, len(plot_base)), random_state=PLOT_SEED).copy()
med = plot_base["novelty"].median()
plot_base["grp"] = np.where(plot_base["novelty"] >= med, "high", "low")

series = []
rel = list(range(WIN[0], WIN[1] + 1))

for r in plot_base.itertuples(index=False):
    code = str(r.security_code).zfill(5)
    g = stock_by.get(code)
    if g is None or g.empty:
        continue
    evdf = event_window_panel(
        stock_df=g,
        rm=rm,
        submit_ts=pd.to_datetime(r.submit_ts),
        window=WIN,
        cutoff_hour=CUTOFF_HOUR,
        est_window=EST_WINDOW,
        min_est_obs=MIN_EST_OBS,
    )
    if evdf is None:
        continue
    evdf = evdf.sort_values("date").reset_index(drop=True)
    if len(evdf) != len(rel):
        continue
    series.append(
        pd.DataFrame(
            {
                "rel": rel,
                "grp": r.grp,
                "abvol_mkt_daily": evdf["avol"].astype(float) if "avol" in evdf.columns else np.nan,
                "abs_ar": np.abs(evdf["ar"].astype(float)),
            }
        )
    )

ser = pd.concat(series, ignore_index=True)
agg = ser.groupby(["grp", "rel"]).mean(numeric_only=True).reset_index()

# 1) abvol (daily)
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for grp in ["low", "high"]:
    s = agg[agg["grp"] == grp]
    ax.plot(s["rel"], s["abvol_mkt_daily"], label=grp)
ax.axvline(0, color="k", linewidth=1)
ax.set_title("market-adjusted log-volume (daily)")
ax.set_xlabel("event time (trading day)")
ax.legend()
plt.tight_layout()

OUT1 = os.path.join(FIG_DIR, "event_time_abvol.png")
fig.savefig(OUT1, dpi=200)
print("saved:", OUT1)
plt.close(fig)

# 2) |AR| (daily)
fig, ax = plt.subplots(1, 1, figsize=(8, 4))
for grp in ["low", "high"]:
    s = agg[agg["grp"] == grp]
    ax.plot(s["rel"], s["abs_ar"], label=grp)
ax.axvline(0, color="k", linewidth=1)
ax.set_title("|AR| (daily, market model)")
ax.set_xlabel("event time (trading day)")
ax.legend()
plt.tight_layout()

OUT2 = os.path.join(FIG_DIR, "event_time_abs_ar.png")
fig.savefig(OUT2, dpi=200)
print("saved:", OUT2)
plt.close(fig)

agg



## スライドへの貼り付け

- 図（PNG）は `slides/figures/` に保存されます。
  - `event_time_abvol.png`
  - `event_time_abs_ar.png`
- 回帰の要約表は `slides/tables/regression_summary.csv` に保存されます。

`slides/final_presentation.md` の「図」スライドにそのまま貼り付ければOKです。

